# 03 — Feature Engineering (2023–2025 data)
**Input:** `data/Cleaned/aqi_2023_2025_clean.csv`  **Output:** `data/Cleaned/aqi_2023_25_features.csv`

## The prediction task
`AQI_Bucket` is computed directly from `AQI`, so predicting *today's* bucket from *today's* AQI is trivial. We predict **tomorrow** instead:

| Task | Target | Metrics |
|---|---|---|
| A. Classification | tomorrow's `AQI_Bucket` | per-class precision, recall, F1, macro-F1, confusion matrix |
| B. Regression | tomorrow's `AQI` | MAE, RMSE, R² |

Your autocorrelation plot in the EDA justifies both: yesterday's AQI carries a lot of information about today's. This file has no pollutant concentrations, so we do **not** predict pollutants.

## Feature groups
| Group | Features |
|---|---|
| calendar | month, season, weekday, is_weekend, is_winter, is_festival (Diwali ±3 days) |
| lags | AQI_lag1, lag2, lag3, lag7 |
| rolling | 3, 7 and 14-day mean and 7-day std, **shifted by one day** so today is not included |
| context | state, number_of_monitoring_stations, today's and yesterday's dominant pollutant |
| city | city statistics computed on training data only (done in notebook 04) |

**Golden rule:** every feature must be known *today*. Nothing from tomorrow may leak in.

In [ ]:
import pandas as pd
import numpy as np
import os, sys
sys.path.append("..")
from src.aqi_common import ALL_FEATURES, FEATURE_GROUPS

pd.set_option('display.max_columns', None)
IN_PATH = "../data/Cleaned/aqi_2023_2025_clean.csv"
OUT_PATH = "../data/Cleaned/aqi_2023_25_features.csv"


## Step 1 — Load and tidy the cleaned file
Works whether your cleaned file kept the raw Kaggle column names or already uses `City`, `AQI`, `AQI_Bucket`. It also tries both date formats.

In [ ]:
raw = pd.read_csv(IN_PATH)
df = raw.rename(columns={'area': 'City', 'aqi_value': 'AQI', 'air_quality_status': 'AQI_Bucket', 'date': 'Date'})

d = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
if d.isna().mean() > 0.5:
    d = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df['Date'] = d
df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')

for col, default in [('state', 'Unknown'), ('number_of_monitoring_stations', np.nan), ('prominent_pollutants', np.nan)]:
    if col not in df.columns:
        df[col] = default

df = df.dropna(subset=['City', 'Date', 'AQI', 'AQI_Bucket'])
df['City'] = df['City'].astype(str).str.strip().str.title()
df['AQI_Bucket'] = df['AQI_Bucket'].astype(str).str.strip().str.title()
df = df.drop_duplicates(subset=['City', 'Date'])

df = df[['City', 'state', 'Date', 'AQI', 'AQI_Bucket', 'number_of_monitoring_stations', 'prominent_pollutants']]
df = df.sort_values(['City', 'Date']).reset_index(drop=True)
print("Rows:", len(df), "| cities:", df['City'].nunique(), "| dates:", df['Date'].min().date(), "to", df['Date'].max().date())
df.head()

## Step 2 — Trap 1: date gaps
`shift(1)` moves by **row**, not by **day**. If a city is missing a day, "lag 1" would silently be 2 days old. Check how many gaps exist.

In [ ]:
gap = df.groupby('City')['Date'].diff().dt.days.dropna()
print("Days between consecutive rows (should mostly be 1):")
print(gap.value_counts().sort_index().head(10))
print("\nShare of consecutive rows exactly 1 day apart:", round((gap == 1).mean() * 100, 2), "%")

## Step 3 — Fix the gaps
We give every city a complete daily calendar. Missing days become empty rows (`AQI = NaN`), so a shift by *k rows* is now exactly *k days*. Any feature that would use a missing day becomes NaN and that row is dropped later, instead of being quietly wrong.

In [ ]:
parts = []
for city, g in df.groupby('City', sort=False):
    idx = pd.date_range(g['Date'].min(), g['Date'].max(), freq='D')
    g = g.set_index('Date').reindex(idx)
    g.index.name = 'Date'
    g['City'] = city
    parts.append(g.reset_index())
full = pd.concat(parts, ignore_index=True)

print("Rows before:", len(df), "| after adding missing days:", len(full))
print("Empty rows added:", full['AQI'].isna().sum())

# static columns: carry the city's value into the empty rows
full['state'] = full.groupby('City')['state'].transform(lambda s: s.ffill().bfill())
st = full.groupby('City')['number_of_monitoring_stations'].transform(lambda s: s.ffill().bfill())
full['number_of_monitoring_stations'] = st.fillna(st.median())

## Step 4 — Calendar features
`is_festival` marks Diwali ±3 days. Your data covers Diwali 2023 and 2024 (2025's is listed for completeness but falls after the data ends).

In [ ]:
def month_to_season(m):
    if m in [12, 1, 2]: return 'Winter'
    if m in [3, 4, 5]: return 'Summer'
    if m in [6, 7, 8, 9]: return 'Monsoon'
    return 'Post-Monsoon'

full['month'] = full['Date'].dt.month
full['season'] = full['month'].apply(month_to_season)
full['is_winter'] = full['month'].isin([11, 12, 1, 2]).astype(int)
full['weekday'] = full['Date'].dt.dayofweek
full['is_weekend'] = (full['weekday'] >= 5).astype(int)

diwali = pd.to_datetime(['2023-11-12', '2024-11-01', '2025-10-20'])
days_apart = np.abs((full['Date'].to_numpy()[:, None] - diwali.to_numpy()[None, :]) / np.timedelta64(1, 'D'))
full['is_festival'] = (days_apart.min(axis=1) <= 3).astype(int)

print("Festival rows:", full['is_festival'].sum())
full[['Date', 'month', 'season', 'is_winter', 'weekday', 'is_weekend', 'is_festival']].head()

## Step 5 — Lag and rolling features
- `AQI` itself is **today's** value (known today, so allowed).
- `AQI_lagK` = AQI K days ago.
- Rolling statistics use `shift(1)` first, so the window ends **yesterday**.
- `min_periods` lets a window with a few missing days still produce a value.

In [ ]:
full = full.sort_values(['City', 'Date']).reset_index(drop=True)
g = full.groupby('City')['AQI']

for k in [1, 2, 3, 7]:
    full[f'AQI_lag{k}'] = g.shift(k)

for w in [3, 7, 14]:
    full[f'AQI_roll{w}'] = g.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=max(2, int(w * 0.7))).mean())
full['AQI_std7'] = g.transform(lambda x: x.shift(1).rolling(7, min_periods=5).std())
full['AQI_change1'] = full['AQI'] - full['AQI_lag1']

full[['City', 'Date', 'AQI', 'AQI_lag1', 'AQI_lag2', 'AQI_lag3', 'AQI_lag7', 'AQI_roll3', 'AQI_roll7', 'AQI_roll14']].head(12)

## Step 6 — Context features
The dominant pollutant is known today, so **today's** and **yesterday's** values are both allowed. We keep the six most common pollutants and group the rest as `Other`.

In [ ]:
first = full['prominent_pollutants'].astype('string').str.split(',').str[0].str.strip()
top = first.value_counts().head(6).index
pol = first.where(first.isin(top), 'Other')
pol = pol.where(first.notna())              # empty (gap) rows stay empty
full['pollutant_today'] = pol
full['pollutant_prev'] = full.groupby('City')['pollutant_today'].shift(1)
full['pollutant_today'] = full['pollutant_today'].fillna('Unknown').astype(str)
full['pollutant_prev'] = full['pollutant_prev'].fillna('Unknown').astype(str)
full['state'] = full['state'].fillna('Unknown').astype(str)

print(full['pollutant_today'].value_counts())

## Step 7 — Targets
`shift(-1)` looks one day **forward**. Targets are the only place that is allowed to look ahead, and they are never used as features.

In [ ]:
full['target_AQI_next'] = full.groupby('City')['AQI'].shift(-1)
full['target_bucket_next'] = full.groupby('City')['AQI_Bucket'].shift(-1)
full[['City', 'Date', 'AQI', 'AQI_Bucket', 'target_AQI_next', 'target_bucket_next']].head(5)

## Step 8 — Keep only complete rows
A row is usable only if today's AQI, all lag and rolling features, and tomorrow's target exist. Rows lost to date gaps are removed here.\n\n*If the number removed looks too large for your data, you can drop `AQI_lag2` and `AQI_lag3` from the `need` list below; the gradient boosting models accept missing values, but Logistic Regression and Ridge do not, so keep them if you want those models.*

In [ ]:
need = ['AQI', 'AQI_lag1', 'AQI_lag2', 'AQI_lag3', 'AQI_lag7',
        'AQI_roll3', 'AQI_roll7', 'AQI_roll14', 'AQI_std7', 'AQI_change1',
        'target_AQI_next', 'target_bucket_next']

before = len(full)
feat = full.dropna(subset=need).copy()
print("Rows before:", before, "| usable rows:", len(feat), "| removed:", before - len(feat))

keep = ['City', 'Date'] + [c for c in ALL_FEATURES if c != 'City'] + ['AQI_Bucket', 'target_AQI_next', 'target_bucket_next']
feat = feat[keep].reset_index(drop=True)
feat.head()

## Step 9 — Trap 2: leakage tests
We rebuild three values by hand from the original data and check they match the feature table exactly:
1. the target is really *tomorrow's* AQI,
2. `AQI_lag1` is really *yesterday's* AQI,
3. `AQI_roll7` really covers the 7 days *before* today.

In [ ]:
city = feat['City'].value_counts().index[0]
s = df[df['City'] == city].set_index('Date')['AQI'].asfreq('D')
chk = feat[feat['City'] == city].sample(min(200, (feat['City'] == city).sum()), random_state=0)
one, seven = pd.Timedelta(days=1), pd.Timedelta(days=7)

ok_target = all(np.isclose(s[d + one], t) for d, t in zip(chk['Date'], chk['target_AQI_next']))
ok_lag1 = all(np.isclose(s[d - one], v) for d, v in zip(chk['Date'], chk['AQI_lag1']))
ok_roll7 = all(np.isclose(s[d - seven:d - one].mean(), v) for d, v in zip(chk['Date'], chk['AQI_roll7']))

print(f"Checked {len(chk)} rows of {city}")
print("target = tomorrow's AQI  :", ok_target)
print("AQI_lag1 = yesterday's   :", ok_lag1)
print("AQI_roll7 excludes today :", ok_roll7)
assert ok_target and ok_lag1 and ok_roll7, "Leakage or alignment problem - do not continue"

assert not any('next' in c for c in ALL_FEATURES), "A feature name contains 'next'"
print("\nNo feature column refers to tomorrow. Features used:", len([c for c in ALL_FEATURES]))

## Step 10 — Sanity checks and the baseline to beat
The simplest forecast is **persistence**: "tomorrow = today". Your models must beat it to be useful.

In [ ]:
print("Target class counts (tomorrow's category):")
print(feat['target_bucket_next'].value_counts(), "\n")

print("Correlation today vs tomorrow:", round(feat['AQI'].corr(feat['target_AQI_next']), 3))
print("Persistence MAE (AQI points) :", round((feat['AQI'] - feat['target_AQI_next']).abs().mean(), 1))
print("Persistence accuracy (bucket):", round((feat['AQI_Bucket'] == feat['target_bucket_next']).mean(), 3))

**Takeaway:** *(write after running: how strong is the day-to-day correlation, and how good is persistence? A high persistence score means the models will have to work hard to add value.)*

## Step 11 — Save the feature table

In [ ]:
os.makedirs("../data/Cleaned", exist_ok=True)
feat.to_csv(OUT_PATH, index=False)
print("Saved", OUT_PATH, "| shape:", feat.shape)
print("Date range:", feat['Date'].min().date(), "to", feat['Date'].max().date())

## Summary
You now have one row per city per day with only information known *today*, plus two targets for tomorrow. Next notebook: `04_modelling_2023_25.ipynb`.

### Commit plan (run each after its step, with the notebook saved)
```bash
git add notebooks/03_feature_engineering_2023_25.ipynb
git commit -m "features: add calendar, lag and rolling features"
git push origin nikita-work

git add notebooks/03_feature_engineering_2023_25.ipynb data/Cleaned/aqi_2023_25_features.csv src/
git commit -m "features: save feature table for 2023-25"
git push origin nikita-work
```
The feature CSV is large; if GitHub or your `.gitignore` objects, leave the CSV out and let teammates regenerate it by running this notebook.